# Code Book — Operational Datasets

**Programme:** Applied Machine Learning for Operations — Model Builder Track
**Prepared for:** Nigerian Communications Satellite Limited (NIGCOMSAT)
**Prepared by:** Publica Academy
**Primary use:** Week 3, Time-Series Fundamentals. Carried forward into Weeks 4–6 and available for capstone scoping.

---

## Provenance — read this first

These are **synthetic teaching datasets**. They are not published NIGCOMSAT operational records, and no figure in them is an actual measurement.

Each series was specified to reproduce the real operating characteristics of the platform — a GEO satellite at 42.5°E, eclipse seasons around the two equinoxes, Ka-band rain attenuation over a tropical rainfall climate, 28 active transponders, and a five-day working week in the Network Operations Centre — so that the techniques taught in the programme land on data that behaves the way your real telemetry behaves.

This is deliberate, and it follows the programme design: teaching runs on a synthetic telemetry set so that realistic satellite data can be used freely from day one, while capstones run on real, accessible operational data tied to a named unit process.

**Working rule:** because these files carry no real operational data, they may be used freely in AI assistants and shared outside the training server. Real NIGCOMSAT telemetry, RF records and network logs remain subject to the data restrictions taught in Week 2 — check before pasting any of it into an AI tool.

## The two dataset families

**A. Satellite telemetry (the spine).** Bus power, battery health, RF link quality and amplifier thermals. These are the files the programme is built around: they carry the trend, seasonality, cycle and noise taught in Week 3, the anomalies used in Weeks 4–5, and the degradation signals forecast in Week 6.

**B. Service and network records.** Link availability, transponder utilisation, VSAT uptime, throughput and NOC tickets. Service-level series with different rhythms — most importantly a **weekly** one driven by the working roster, which telemetry does not have.

Both families are needed. Telemetry teaches physical seasonality (eclipse seasons, diurnal cycles); service records teach human seasonality (the working week). Week 3 uses both to make the point that a seasonal period comes from the process, not from a default setting.

### Dataset index

| File | Family | Cadence | Span | Chief teaching use |
|---|---|---|---|---|
| `bus_power_telemetry_hourly.csv` | A | hourly | 120 days | Eclipse-season cycle, anomalies, period=24 |
| `battery_health_daily.csv` | A | daily | 3 years | Degradation trend, twice-yearly seasonality |
| `rf_link_quality_hourly.csv` | A | hourly | Aug 2025 | Rain fade, mean vs minimum, SLA breach |
| `twta_thermal_daily.csv` | A | daily | 2 years | Slow drift against a yearly cycle |
| `ka_band_availability_daily.csv` | B | daily | 3 years | The anchor series: all four components |
| `ka_band_availability_monthly.csv` | B | monthly | 3 years | Decomposition with period=12 |
| `transponder_utilisation_monthly.csv` | B | monthly | 3 years | Saturating trend |
| `noc_tickets_daily.csv` | B | daily | 2 years | Weekly rhythm, rolling windows, period=7 |
| `noc_tickets_weekly.csv` | B | weekly | 1 year | Noise, and what aggregation removes |
| `vsat_uptime_by_region_weekly.csv` | B | weekly | 1 year | Long format, small multiples, indexing |
| `bandwidth_throughput_daily.csv` | B | daily | 2 years | Autocorrelation at lag 7 |
| `additive_multiplicative_demo.csv` | — | — | 48 steps | Additive vs multiplicative, teaching aid |

### Where each file is used across the eight weeks

| File | Wk 2 cleaning | Wk 3 time series | Wk 4–5 anomalies | Wk 6 forecasting | Capstone brief it rehearses |
|---|---|---|---|---|---|
| `bus_power_telemetry_hourly` | ✓ | ✓ | ✓ | | Telemetry anomaly detection |
| `battery_health_daily` | | ✓ | ✓ | ✓ | Power and battery degradation |
| `rf_link_quality_hourly` | ✓ | ✓ | ✓ | | RF link-quality prediction |
| `twta_thermal_daily` | | ✓ | ✓ | ✓ | Telemetry anomaly / degradation |
| `ka_band_availability_daily` | ✓ | ✓ | ✓ | ✓ | Carrier and capacity monitoring |
| `transponder_utilisation_monthly` | | ✓ | | ✓ | Carrier and capacity monitoring |
| `noc_tickets_daily` | ✓ | ✓ | | ✓ | Site-downtime prediction |
| `vsat_uptime_by_region_weekly` | ✓ | ✓ | ✓ | | Site-downtime prediction |
| `bandwidth_throughput_daily` | | ✓ | | ✓ | Carrier and capacity monitoring |

Anomalies are deliberately embedded in the telemetry files. Week 3 finds them as large residuals; Week 4 detects them systematically. The answer key is in each dataset section below — hold it back from participants until after the exercise.

## Conventions used in every file

| Item | Convention |
|---|---|
| Encoding | UTF-8, comma-separated, one header row |
| Dates | ISO 8601 — `YYYY-MM-DD` (daily), `YYYY-MM-DDTHH:00` (hourly), `YYYY-MM` (monthly) |
| Time zone | UTC throughout. WAT is UTC+1 all year, with no daylight saving |
| Missing values | Empty field — never `-1`, `0` or `NULL` |
| Percentages | 0–100 scale, not 0–1 |
| Flags | Integer `0`/`1`, not `True`/`False` |
| Sort order | Ascending by timestamp; `vsat_uptime_by_region_weekly` is timestamp then region |
| Units | Stated in the column name where ambiguity is possible (`_pct`, `_db`, `_a`, `_v`, `_c`, `_gbps`) |

## Setup

Run this cell first. It assumes the notebook sits next to the `data/` folder.

**What this cell does.** Imports pandas and matplotlib, points `DATA` at the `data/` folder beside this notebook, widens the console display so wide frames print readably, and lists every CSV present. Run it first — everything below depends on it. If the file list comes back empty, the notebook is not sitting next to `data/`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path("data")
pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 25)

sorted(p.name for p in DATA.glob("*.csv"))

## A reusable profiling helper

Every dataset section below calls this. It is also the check to run on your **own** exports before analysing them — it answers the Week 2 questions (gaps, duplicates, sampling rate) and the Week 3 question (what range and spread am I dealing with).

**What this cell does.** Defines the `profile()` function used by every dataset section below. Given a frame and its time column it prints the row count, column names, first and last timestamp, duplicated timestamps, missing cells, and — when you pass `freq` — how many expected periods are absent. It ends with `describe()` so you can see each column's range. Output is meant to be read, not captured. Pass `group_col` for the one long-format file, where duplicates must be counted per group.

In [ ]:
def profile(df, time_col, freq=None, group_col=None):
    """Print a short structural profile of a time-series DataFrame."""
    t = pd.to_datetime(df[time_col])
    print(f"rows          : {len(df):,}")
    print(f"columns       : {list(df.columns)}")
    print(f"range         : {t.min()}  ->  {t.max()}")
    if group_col:
        print(f"groups        : {df[group_col].nunique()} ({', '.join(sorted(df[group_col].unique()))})")
        dup = df.duplicated(subset=[time_col, group_col]).sum()
    else:
        dup = t.duplicated().sum()
    print(f"duplicates    : {dup}")
    print(f"missing cells : {int(df.isna().sum().sum())}")
    if freq and not group_col:
        expected = pd.date_range(t.min(), t.max(), freq=freq)
        print(f"expected rows : {len(expected):,} at freq={freq}")
        print(f"gaps          : {len(expected.difference(t))}")
    print()
    print(df.describe(include="number").round(3))

---

# Family A — Satellite telemetry

The four files in this family are the programme spine. They come from the platform itself rather than from the service it delivers, which means their seasonality is physical: eclipse seasons set by the orbit, rainfall set by the climate, sun angle set by the calendar. None of them has a weekly rhythm — that arrives with Family B.

---

## A1. Bus power and battery telemetry — hourly

`data/bus_power_telemetry_hourly.csv`

Electrical power subsystem telemetry across a full eclipse season. A GEO satellite in sunlight runs entirely off its solar arrays; twice a year, around the equinoxes, the Earth's shadow crosses the orbit and the satellite runs on battery for up to 72 minutes a day. This file captures that cycle at hourly resolution, which makes it the clearest example in the collection of a **daily period with a seasonal envelope**.

**Shape:** 2,880 rows — 2025-08-01T00:00 to 2025-11-28T23:00, one row per hour

### Columns

| Column | Type | Unit | Description |
|---|---|---|---|
| `timestamp` | datetime (hourly) | — | Hour of the reading, UTC |
| `solar_array_current_a` | float | amperes | Total array output. Around 27.8 A in sunlight, near zero in eclipse, elevated during post-eclipse recharge |
| `bus_voltage_v` | float | volts | Regulated bus voltage. About 50.4 V nominal, sagging to roughly 46 V on battery |
| `battery_soc_pct` | float | percent | Battery state of charge, 0–100. Falls to about 77% at the deepest eclipse |
| `battery_temp_c` | float | °C | Battery pack temperature. About 18 °C nominal, rising during discharge and recharge |
| `eclipse_flag` | int | 0 or 1 | 1 when the satellite is in eclipse during that hour |

### Notes and known behaviour

- **Eclipse season runs 2025-09-01 to 2025-10-16** in this file — 46 eclipse hours in total. Outside those dates the platform is in continuous sunlight and the series is nearly flat.
- Eclipse occurs at **21:00 UTC**, close to satellite local midnight at 42.5°E. This is why the series has a 24-hour period: use `period=24` when decomposing, and expect an autocorrelation spike at lag 24.
- Discharge depth tracks eclipse duration, so the season builds and fades rather than switching on. That envelope is the seasonal component; the daily dropout is the period.
- **Anomaly answer key** — hold back from participants. Four injected events: a partial array-string dropout at hour 417 (2025-08-18T09:00), a two-hour dropout with thermal excursion at hours 1406–1407 (2025-09-28T14:00–15:00), and a further dropout at hour 2187 (2025-10-31T03:00). Each shows array current at roughly 55% of expected with battery temperature about 4.6 °C high.
- The flat sunlit baseline makes this the best file in the collection for control limits in Week 4: normal behaviour is genuinely stable, so deviations are unambiguous.

**What to read in the output.** 2,880 hourly rows with no gaps and no duplicates. The two printed lines below the profile are the important ones: 46 eclipse hours, falling between 1 September and 16 October. Check `battery_soc_pct` in `describe()` — the minimum should sit near 77%, which is the deepest discharge of the season, while the mean stays close to 100% because most hours are in full sunlight.

In [ ]:
bus = pd.read_csv(DATA / "bus_power_telemetry_hourly.csv",
                  parse_dates=["timestamp"], index_col="timestamp").asfreq("H")

profile(bus.reset_index(), "timestamp", freq="H")

print("\neclipse hours:", int(bus["eclipse_flag"].sum()))
print("eclipse season:", bus.index[bus["eclipse_flag"] == 1].min().date(),
      "->", bus.index[bus["eclipse_flag"] == 1].max().date())

---

## A2. Battery health — daily

`data/battery_health_daily.csv`

Three years of daily battery condition, built for the degradation question: is the battery ageing faster than planned? It carries a genuine downward trend in capacity, and — unusually — **two seasonal peaks a year**, because eclipse seasons fall around both equinoxes. It is the best argument in the collection against assuming seasonality means one annual hump.

**Shape:** 1,096 rows — 2023-01-01 to 2025-12-31, one row per day

### Columns

| Column | Type | Unit | Description |
|---|---|---|---|
| `date` | date | — | Calendar day |
| `eclipse_minutes` | float | minutes | Eclipse duration that day. 0 outside eclipse seasons, up to 72 at the equinox |
| `depth_of_discharge_pct` | float | percent | How far the battery was drawn down. Near zero outside eclipse seasons |
| `eod_voltage_v` | float | volts | End-of-discharge voltage. Falls with both discharge depth and age |
| `est_capacity_ah` | float | amp-hours | Estimated usable capacity. Fades from about 103 Ah to about 97 Ah across the three years |
| `cycle_count` | int | count | Cumulative charge/discharge cycles, starting from 4,180 at day one |

### Notes and known behaviour

- **Two seasonal peaks a year**, centred on roughly 21 March and 23 September, each about 45 days wide. Decompose the monthly aggregate with `period=12` and the seasonal profile will show both.
- **Capacity fade is about −2.1% a year** plus a small per-cycle wear term. Over three years that is roughly 6 Ah, which is the trend Week 6 forecasts to a threshold.
- `eod_voltage_v` mixes two effects: discharge depth (seasonal) and ageing (trend). It is the best single column for practising decomposition, because both components are real and separable.
- Between eclipse seasons `depth_of_discharge_pct` sits near 0.4–0.9% rather than exactly zero, representing housekeeping load. Do not treat those as missing.
- No anomalies are injected in this file. The degradation is smooth and monotonic by design, so Week 6 can be about extrapolation rather than outlier handling.

**What to read in the output.** 1,096 daily rows. The first plot is the point of the file: eclipse minutes rise and fall **twice** a year, around each equinox, rather than once. The second plot shows capacity fading from roughly 103 Ah to 97 Ah — the degradation trend Week 6 extrapolates. Note in `describe()` that `depth_of_discharge_pct` has a low mean but a high maximum: most days are quiet, eclipse-season days are not.

In [ ]:
bat = pd.read_csv(DATA / "battery_health_daily.csv",
                  parse_dates=["date"], index_col="date").asfreq("D")

profile(bat.reset_index(), "date", freq="D")

# the twice-yearly eclipse seasons, visible as two humps
bat["eclipse_minutes"].plot(figsize=(11, 2.6), color="#0E7C86", lw=1.2)
plt.title("Eclipse minutes per day — two seasons a year")
plt.show()

bat["est_capacity_ah"].plot(figsize=(11, 2.6), color="#C0504D", lw=1.6)
plt.title("Estimated battery capacity — the degradation trend")
plt.show()

---

## A3. RF link quality — hourly, August 2025

`data/rf_link_quality_hourly.csv`

One month of Ka-band link telemetry at the wettest point of the Nigerian rainy season. Ka-band is the most rain-sensitive band on the platform, so this file is where rain fade is visible as a physical process rather than a monthly average. It is also the file that demonstrates why an SLA cannot be reported from a daily mean.

**Shape:** 744 rows — 2025-08-01T00:00 to 2025-08-31T23:00, one row per hour

### Columns

| Column | Type | Unit | Description |
|---|---|---|---|
| `timestamp` | datetime (hourly) | — | Hour of the reading, UTC |
| `rain_rate_mmhr` | float | mm/hr | Rain rate at the earth station. 0 in most hours, up to about 58 in heavy convective storms |
| `cn_ratio_db` | float | dB | Carrier-to-noise ratio at the receiver. About 14.25 dB in clear sky |
| `uplink_power_dbm` | float | dBm | Transmitted uplink power. Rises under uplink power control, with about 6 dB of headroom |
| `link_margin_db` | float | dB | `cn_ratio_db` minus the 8.5 dB demodulation threshold. Negative means the link is down |

### Notes and known behaviour

- **Rain attenuation follows a power law** in rain rate, as in the ITU model. Heavy storms remove far more signal than the 6 dB of uplink power control can restore.
- **56 of 744 hours carry rain. 16 hours fall below 2 dB of margin, and 12 hours go negative** — those twelve are genuine outage hours and are the correct answer to "did we breach?".
- **Diurnal pattern:** storm probability peaks in the mid-afternoon, matching convective rainfall over Nigeria. Expect a 24-hour component, weaker than the one in the bus power file.
- `uplink_power_dbm` is a *response* to rain, not an independent measurement. It is the clearest example in the collection of two columns that correlate because one causes the other — useful when Week 3 warns against reading correlation as cause.
- Compare `.resample("D").mean()` with `.resample("D").min()`. The mean stays comfortably positive on every day of the month, including days with outage hours. That gap is the whole lesson.

**What to read in the output.** 744 hourly rows for August. The three printed counts are the SLA story: 56 hours carry rain, 16 drop below 2 dB of margin, and 12 go negative — those twelve are outage hours. Then compare the `mean` and `min` columns in the daily table: the mean never looks alarming, even on days containing an outage. That gap is why availability is reported from minimums, not averages.

In [ ]:
rf = pd.read_csv(DATA / "rf_link_quality_hourly.csv",
                 parse_dates=["timestamp"], index_col="timestamp").asfreq("H")

profile(rf.reset_index(), "timestamp", freq="H")

print("\nhours with rain      :", int((rf["rain_rate_mmhr"] > 0).sum()))
print("hours below 2 dB     :", int((rf["link_margin_db"] < 2).sum()))
print("hours below 0 dB     :", int((rf["link_margin_db"] < 0).sum()))
print("\ndaily mean vs daily minimum margin:")
rf["link_margin_db"].resample("D").agg(["mean", "min"]).round(2).head(10)

---

## A4. TWTA thermal — daily

`data/twta_thermal_daily.csv`

Two years of travelling-wave tube amplifier temperature, with the RF drive level that partly explains it. A slow upward drift sits underneath a yearly sun-angle cycle and a rising drive level — three causes in one column. It is the hardest of the four telemetry files to interpret, and the most representative of real subsystem data.

**Shape:** 730 rows — 2024-01-01 to 2025-12-30, one row per day

### Columns

| Column | Type | Unit | Description |
|---|---|---|---|
| `date` | date | — | Calendar day |
| `twta_temp_c` | float | °C | Amplifier baseplate temperature, roughly 59–66 |
| `rf_drive_pct` | float | percent | RF drive level as a share of saturation, rising from about 68% to about 82% |

### Notes and known behaviour

- **Three components in one column.** A slow ageing drift of roughly +0.003 °C a day, a yearly sun-angle cycle of about ±1.6 °C, and a temperature contribution from the rising drive level.
- Because drive level is itself trending, the naive reading — "the amplifier is overheating" — confuses ageing with increased load. Regress out `rf_drive_pct` first, or decompose and compare.
- This is the right file for the Week 3 warning about two trending series: `twta_temp_c` and `rf_drive_pct` correlate strongly, and most of that correlation is shared trend.
- No injected anomalies. The interpretive difficulty is the exercise.

**What to read in the output.** 730 daily rows. The two correlations at the end are the lesson: raw correlation between amplifier temperature and RF drive is high, and most of it is shared trend rather than a physical relationship. After differencing — comparing day-to-day *changes* instead of levels — the correlation collapses. Run this before believing any correlation between two operational series.

In [ ]:
twta = pd.read_csv(DATA / "twta_thermal_daily.csv",
                   parse_dates=["date"], index_col="date").asfreq("D")

profile(twta.reset_index(), "date", freq="D")

# the trap: these two correlate, mostly because both are trending
print("\nraw correlation      :", round(twta["twta_temp_c"].corr(twta["rf_drive_pct"]), 3))
print("after differencing   :", round(twta["twta_temp_c"].diff().corr(twta["rf_drive_pct"].diff()), 3))

---

# Family B — Service and network records

These files describe the service the platform delivers rather than the platform itself. Their seasonality is partly human: the Network Operations Centre works a five-day week, so ticket and throughput series carry a seven-day rhythm that no telemetry file has. Week 3 needs both kinds to make the point that the seasonal period comes from the process.

---

## B1. Ka-band link availability — daily

`data/ka_band_availability_daily.csv`

The anchor series for Week 3. Daily availability of the Ka-band Nigeria spot beam, as the percentage of the day the link met its performance threshold. It is the one file that shows all four components — trend, seasonality, cycle and noise — clearly enough to point at each in turn.

**Shape:** 1,096 rows — 2023-01-01 to 2025-12-31, one row per day

### Columns

| Column | Type | Unit | Description |
|---|---|---|---|
| `timestamp` | date | — | Calendar day of the reading, UTC |
| `availability_pct` | float | percent | Share of the day the link met threshold, roughly 94–100 |
| `band` | string | — | Always `Ka` — present so the schema matches a multi-band export |
| `beam` | string | — | Always `Nigeria spot beam` |

### Notes and known behaviour

- **Seasonal trough is August**, with the wet season running roughly April to October — the same climate driver as the RF file, seen at service level.
- **Trend** is about +0.05 points a year, representing progressive ground-station upgrades.
- Sharp single-day drops are rain-fade events, generated more often in the wet season. They are real events, not outliers to delete.
- Deliberately clean: no gaps, no duplicated dates. Week 3 teaches the checks on data that passes them, so that failures in real exports stand out by contrast.

**What to read in the output.** 1,096 daily rows, no gaps, no duplicates. This is the anchor series for the week, so confirm the range in `describe()`: availability sits mostly above 99% with a minimum in the mid-90s, and those low days are rain-fade events rather than data errors. `head()` shows the two constant columns (`band`, `beam`) that exist so the schema matches a real multi-band export.

In [ ]:
ka = pd.read_csv(DATA / "ka_band_availability_daily.csv",
                 parse_dates=["timestamp"], index_col="timestamp").asfreq("D")

profile(ka.reset_index(), "timestamp", freq="D")
ka.head()

---

## B2. Ka-band link availability — monthly

`data/ka_band_availability_monthly.csv`

The monthly mean of B1, supplied ready-made so decomposition exercises do not depend on a resample step. Thirty-six months is exactly three seasonal cycles — the practical minimum for trusting a seasonal profile, and worth stating whenever you report one built from it.

**Shape:** 36 rows — 2023-01 to 2025-12, one row per month

### Columns

| Column | Type | Unit | Description |
|---|---|---|---|
| `month` | string `YYYY-MM` | — | Calendar month. Parsed, it becomes the first of the month |
| `availability_pct` | float | percent | Monthly mean availability, roughly 97.6–100 |

### Notes and known behaviour

- Use `period=12` when decomposing this file.
- Three cycles is the minimum, not a comfortable margin. Compare with `battery_health_daily`, which gives six eclipse seasons over the same span.

**What to read in the output.** 36 monthly rows — exactly three seasonal cycles. The plot is the shape the whole week refers back to: a slow upward drift with a deep dip every wet season. Use this file whenever an exercise calls for `period=12`, and state the three-cycle limitation whenever you report a seasonal profile built from it.

In [ ]:
kam = pd.read_csv(DATA / "ka_band_availability_monthly.csv",
                  parse_dates=["month"], index_col="month").asfreq("MS")

profile(kam.reset_index(), "month", freq="MS")
kam["availability_pct"].plot(figsize=(11, 2.8), color="#0E7C86", lw=2)
plt.title("Ka-band availability, monthly mean")
plt.show()

---

## B3. Transponder utilisation — monthly

`data/transponder_utilisation_monthly.csv`

Utilisation of leased capacity across the 28 active transponders. The clean example of a **trend**: the series climbs from roughly half to four-fifths of capacity as leases are sold, then begins to flatten.

**Shape:** 36 rows — 2023-01 to 2025-12

### Columns

| Column | Type | Unit | Description |
|---|---|---|---|
| `month` | string `YYYY-MM` | — | Calendar month |
| `util_pct` | float | percent | Share of leased capacity in use, roughly 50–85 |
| `active_transponders` | int | count | Always 28 |

### Notes and known behaviour

- The trend is a **saturating curve, not a straight line** — deliberately, so the discussion does not collapse into "fit a line".
- A mild seasonal component of about ±1.6 points sits on top.
- With `active_transponders` constant, the column exists only to show a static field carried through a real export.

**What to read in the output.** 36 monthly rows. Read the plot as a trend, not as a wobble: utilisation climbs from about 50% into the low 80s and then begins to flatten. That flattening is why "fit a straight line" is the wrong instinct here. `active_transponders` is constant at 28 — a static field carried through a real export.

In [ ]:
util = pd.read_csv(DATA / "transponder_utilisation_monthly.csv",
                   parse_dates=["month"], index_col="month").asfreq("MS")

profile(util.reset_index(), "month", freq="MS")
util["util_pct"].plot(figsize=(11, 2.8), color="#0E7C86", lw=2)
plt.title("Transponder utilisation, % of leased capacity")
plt.show()

---

## B4. NOC fault tickets — daily

`data/noc_tickets_daily.csv`

Fault tickets raised per day in the Network Operations Centre. The workhorse for rolling windows, because it carries two rhythms at once: a five-day working week and a wet-season rise. It is the clearest demonstration of why a seven-day window is the natural first choice.

**Shape:** 730 rows — 2024-01-01 to 2025-12-30

### Columns

| Column | Type | Unit | Description |
|---|---|---|---|
| `date` | date | — | Calendar day |
| `tickets_raised` | int | count | All tickets opened that day, roughly 0–20 |
| `priority_1` | int | count | Subset of `tickets_raised` classed priority 1 |

### Notes and known behaviour

- **Weekly pattern:** weekends run at roughly half of weekday volume. A 7-day window removes it exactly.
- **Seasonal pattern:** volume rises through the wet season, peaking in August, matching the availability and RF files.
- `priority_1` is a noisy 18% share of the total, so it is far sparser — useful for showing why small counts need wider windows.
- **Decompose with `period=7`, not 12.** This is the file that teaches the period is a property of the process.

**What to read in the output.** 730 daily rows. The table at the end is the reason this file exists: weekday means sit well above weekend means. That gap is why a seven-day rolling window is the natural first choice, and why this file must be decomposed with `period=7` rather than 12. Compare the `priority_1` column in `describe()` — far sparser, which is why small counts need wider windows.

In [ ]:
tk = pd.read_csv(DATA / "noc_tickets_daily.csv",
                 parse_dates=["date"], index_col="date").asfreq("D")

profile(tk.reset_index(), "date", freq="D")

# the weekly rhythm this file exists to demonstrate
tk.groupby(tk.index.day_name())["tickets_raised"].mean().round(2)

---

## B5. NOC fault tickets — weekly

`data/noc_tickets_weekly.csv`

Weekly ticket totals for one year. The example of **noise**: week-to-week movement of a few tickets either way that carries no operational meaning, and should not trigger action.

**Shape:** 52 rows — weeks beginning 2025-01-06

### Columns

| Column | Type | Unit | Description |
|---|---|---|---|
| `week_start` | date | — | Monday that begins the week |
| `tickets_raised` | int | count | Tickets opened during that week, roughly 25–50 |

### Notes and known behaviour

- Weekly aggregation has already removed the day-of-week effect — comparing this with B4 shows what an aggregation step silently does.
- The long-run mean is about 38 tickets a week. Use it to make the point that 38 and 41 are the same week twice.

**What to read in the output.** 52 weekly rows. The dashed line is the long-run mean of about 38 tickets. Most weeks sit within a handful of tickets of it — that scatter is noise, and moving from 38 to 41 is not a change worth investigating. Comparing this plot with the daily file shows what a weekly aggregation silently removes: the entire day-of-week effect.

In [ ]:
tkw = pd.read_csv(DATA / "noc_tickets_weekly.csv",
                  parse_dates=["week_start"], index_col="week_start")

profile(tkw.reset_index(), "week_start", freq="W-MON")
tkw["tickets_raised"].plot(figsize=(11, 2.8), color="#0E7C86", lw=1.6)
plt.axhline(tkw["tickets_raised"].mean(), color="#C0504D", ls="--")
plt.title("NOC tickets per week, with the long-run mean")
plt.show()

---

## B6. VSAT terminal uptime by region — weekly

`data/vsat_uptime_by_region_weekly.csv`

Weekly uptime for VSAT terminals grouped by Nigeria's six geopolitical zones. The only **long-format** file in the collection: one row per region per week. Used for small multiples and for indexing to a baseline.

**Shape:** 312 rows — 52 weeks × 6 regions, 2025

### Columns

| Column | Type | Unit | Description |
|---|---|---|---|
| `week_start` | date | — | Monday that begins the week |
| `region` | string | — | North West, North East, North Central, South West, South East, or South South |
| `uptime_pct` | float | percent | Mean terminal uptime for that region and week, roughly 96–100 |

### Notes and known behaviour

- **Long format** — pivot before plotting: `df.pivot(index="week_start", columns="region", values="uptime_pct")`.
- Southern zones carry a heavier wet-season penalty (about 1.5 points against 0.6 in the north), reflecting the real rainfall gradient.
- South South starts lowest **and** falls furthest — two facts a raw chart cannot separate, which is why the exercise indexes to week 1.
- Pass `group_col="region"` to `profile()`: duplicates must be judged per region, not per week.

**What to read in the output.** 312 rows — 52 weeks across 6 regions. Note that `profile()` is called with `group_col="region"` here, so duplicates are counted per region rather than per week. The `pivot` turns the long format into the wide shape needed for plotting, one column per region. In `describe()`, compare the means and minimums across zones: southern regions sit lower and swing further.

In [ ]:
vsat = pd.read_csv(DATA / "vsat_uptime_by_region_weekly.csv",
                   parse_dates=["week_start"])

profile(vsat, "week_start", group_col="region")

wide = vsat.pivot(index="week_start", columns="region", values="uptime_pct")
wide.describe().round(2)

---

## B7. Bandwidth throughput — daily

`data/bandwidth_throughput_daily.csv`

Daily aggregate throughput across the Nigeria spot beam. Built for autocorrelation: a pronounced working-week cycle produces textbook correlogram spikes at lags 7, 14, 21 and 28.

**Shape:** 730 rows — 2024-01-01 to 2025-12-30

### Columns

| Column | Type | Unit | Description |
|---|---|---|---|
| `date` | date | — | Calendar day |
| `throughput_gbps` | float | Gbps | Aggregate daily throughput, roughly 2.3–3.9 |

### Notes and known behaviour

- **Day-of-week effect:** weekends run at roughly 78–83% of weekday throughput. This is what produces the lag-7 autocorrelation.
- **Trend:** about +0.6 Gbps over the two years, as capacity is taken up.
- **Annual component:** a mild ±5% yearly swing.
- Autocorrelation at lag 1 is around 0.83, and lag 7 is comparable or higher — the practical demonstration that "same weekday last week" is a legitimate forecast baseline for Week 6.

**What to read in the output.** 730 daily rows. The four autocorrelation figures are the point: lag 1 is high because consecutive days resemble each other, and lag 7 is as high or higher because the working week repeats. Lag 30 is weaker. This is the numeric form of the correlogram, and the evidence that "same weekday last week" is a legitimate forecast baseline.

In [ ]:
bw = pd.read_csv(DATA / "bandwidth_throughput_daily.csv",
                 parse_dates=["date"], index_col="date").asfreq("D")

profile(bw.reset_index(), "date", freq="D")

s = bw["throughput_gbps"]
for lag in (1, 7, 14, 30):
    print(f"autocorr(lag={lag:>2}) = {s.autocorr(lag=lag):.3f}")

---

## C1. Additive vs multiplicative — teaching aid

`data/additive_multiplicative_demo.csv`

Not an operational series. Two synthetic columns built from the same trend and the same seasonal shape, combined additively in one and multiplicatively in the other, so the difference can be shown side by side.

**Shape:** 48 rows — month index 0 to 47, no calendar dates

### Columns

| Column | Type | Unit | Description |
|---|---|---|---|
| `month_index` | int | — | Sequence position, 0-based. Deliberately not a date |
| `additive` | float | arbitrary | trend + season + noise — the swing keeps a constant size |
| `multiplicative` | float | arbitrary | trend × season × noise — the swing grows with the level |

### Notes and known behaviour

- Units are arbitrary. This file exists to be looked at, not measured.
- Take logs of the `multiplicative` column and it becomes additive.
- Real counterparts in this collection: `est_capacity_ah` and `availability_pct` behave additively; `tickets_raised` and `throughput_gbps` behave multiplicatively.

**What to read in the output.** Two panels on a shared y-axis. Left: the seasonal swing stays the same size as the level rises. Right: the swing grows with the level. Nothing here is operational — the units are arbitrary and the file exists only to make that contrast visible before you have to choose `model="additive"` or `model="multiplicative"`.

In [ ]:
demo = pd.read_csv(DATA / "additive_multiplicative_demo.csv", index_col="month_index")

fig, axes = plt.subplots(1, 2, figsize=(12, 3), sharey=True)
demo["additive"].plot(ax=axes[0], color="#0E7C86", lw=2, title="Additive")
demo["multiplicative"].plot(ax=axes[1], color="#0E7C86", lw=2, title="Multiplicative")
plt.show()

---

## Seasonal periods at a glance

The single most common mistake with these files is using the wrong period. It is a property of the process, not a default.

| File | Column to decompose | Period | Why |
|---|---|---|---|
| `bus_power_telemetry_hourly` | `battery_soc_pct` | 24 | Eclipse occurs once per day, at 21:00 UTC |
| `rf_link_quality_hourly` | `link_margin_db` | 24 | Convective rainfall peaks mid-afternoon |
| `battery_health_daily` (monthly mean) | `depth_of_discharge_pct` | 12 | Two eclipse seasons a year, at the equinoxes |
| `twta_thermal_daily` (monthly mean) | `twta_temp_c` | 12 | Yearly sun-angle cycle |
| `ka_band_availability_monthly` | `availability_pct` | 12 | Annual wet season |
| `noc_tickets_daily` | `tickets_raised` | 7 | Five-day working week |
| `bandwidth_throughput_daily` | `throughput_gbps` | 7 | Five-day working week |

Confirm the period from a correlogram rather than assuming it — that is what the autocorrelation session is for.

## Using these files with your own data

The schemas above are the contract. To swap in a real NIGCOMSAT export:

1. **Match the column names**, or rename on load with `df.rename(columns={...})`.
2. **Keep the time column ISO-formatted and in one time zone.** If your export is in WAT, either convert it to UTC or convert this data to WAT — do not mix. Telemetry is usually UTC; ticket systems are usually local.
3. **Declare the frequency** with `.asfreq()` after setting the index, so real gaps appear as `NaN` instead of disappearing.
4. **Run `quick_check()`** before anything else. Gaps, duplicated timestamps and a changed sampling rate are all normal in production exports, and all silently break the analysis.
5. **Establish the seasonal period** from a correlogram, not from the table above.
6. **Check the data restriction before using an AI assistant on it.** These synthetic files are unrestricted; your real telemetry may not be.

### One check to run on any new export

**What this cell does.** A compact version of `profile()` that returns a dictionary instead of printing, so you can run it across many files or store the result. It reports row count, date range, duplicated timestamps, missing periods against the frequency you declare, null cells, and whether timestamps ascend. Run it on any new export before analysing it — the last three fields are where production data usually fails.

In [ ]:
def quick_check(path, time_col, freq):
    df = pd.read_csv(path, parse_dates=[time_col])
    t = df[time_col]
    expected = pd.date_range(t.min(), t.max(), freq=freq)
    return {
        "rows": len(df),
        "range": (str(t.min()), str(t.max())),
        "duplicated_timestamps": int(t.duplicated().sum()),
        "missing_periods": len(expected.difference(t)),
        "null_cells": int(df.isna().sum().sum()),
        "monotonic": bool(t.is_monotonic_increasing),
    }

quick_check(DATA / "bus_power_telemetry_hourly.csv", "timestamp", "H")

---

## Change log

| Date | Change |
|---|---|
| 2026-09-02 | First issue. Nine service and network datasets. |
| 2026-09-03 | Added the satellite telemetry family (bus power, battery health, RF link quality, TWTA thermal) so the collection matches the curriculum spine and carries into Weeks 4–6. Restructured into Family A / Family B, added the week-by-week usage map, the seasonal-period table and the anomaly answer key. |

*Applied Machine Learning for Operations, Model Builder Track — prepared for NIGCOMSAT by Publica Academy.*